# Парсинг с помощью BeautifulSoup

Beautiful Soup - это библиотека для Python, которая позволяет парсить (анализировать) HTML и XML документы. Она предоставляет удобный способ искать, навигировать, и модифицировать дерево DOM (Document Object Model), представляющее HTML/XML документ.

In [1]:
# !pip install -U beautifulsoup4

In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [3]:
url = 'http://quotes.toscrape.com/'

In [4]:
r = requests.get(url, verify=False, timeout=10)
r

<Response [200]>

In [5]:
soup = BeautifulSoup(r.content, 'html.parser')


In [6]:
divs = soup.find_all('div', attrs={'class':'quote'})
divs[0]

<div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
<span class="text" itemprop="text">“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”</span>
<span>by <small class="author" itemprop="author">Albert Einstein</small>
<a href="/author/Albert-Einstein">(about)</a>
</span>
<div class="tags">
            Tags:
            <meta class="keywords" content="change,deep-thoughts,thinking,world" itemprop="keywords"/>
<a class="tag" href="/tag/change/page/1/">change</a>
<a class="tag" href="/tag/deep-thoughts/page/1/">deep-thoughts</a>
<a class="tag" href="/tag/thinking/page/1/">thinking</a>
<a class="tag" href="/tag/world/page/1/">world</a>
</div>
</div>

In [7]:
df = pd.DataFrame({
    'text':[],
    'author':[],
    'author_url':[],
    'tags':[],
})

In [8]:
divs[1].find('span', class_='text').text

'“It is our choices, Harry, that show what we truly are, far more than our abilities.”'

In [9]:
def get_quotes(soup, data=df):
    
    divs = soup.find_all('div', attrs={'class':'quote'})
    
    for div in divs:
        quote = {}
        quote['text'] = div.find('span', class_='text').text
        quote['author'] = div.find('small', class_='author').text
        quote['author_url'] = div.find_all('span')[1].find('a').get('href')
        tags = []
        for a_tag in div.find('div', class_='tags').find_all('a', class_='tag'):
            tags.append(a_tag.text)
        quote['tags'] = ', '.join(tags)
        df.loc[len(df)] = quote

In [10]:
url = 'https://quotes.toscrape.com/page/'

for i in range(1, 21):
    print(f'Page N:{i}')
    r = requests.get(url + str(i), timeout=10)
    soup = BeautifulSoup(r.content, 'html.parser')
    
    get_quotes(soup, df)
    
    if 'No quotes found!' in soup.text:
        break

Page N:1


/tmp/ipykernel_57673/1003673812.py:14: Pandas4Warning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. The existing dtype float64 cannot hold the value being set, so the result has dtype str. Cast the object to str before this operation to retain the current behavior.
  df.loc[len(df)] = quote


Page N:2
Page N:3
Page N:4
Page N:5
Page N:6
Page N:7
Page N:8
Page N:9
Page N:10
Page N:11


In [11]:
df.tail()

,text,author,author_url,tags
95,“You never really understand a person until yo...,Harper Lee,/author/Harper-Lee,better-life-empathy
96,“You have to write the book that wants to be w...,Madeleine L'Engle,/author/Madeleine-LEngle,"books, children, difficult, grown-ups, write, ..."
97,“Never tell the truth to people who are not wo...,Mark Twain,/author/Mark-Twain,truth
98,"“A person's a person, no matter how small.”",Dr. Seuss,/author/Dr-Seuss,inspirational
99,“... a mind needs books as a sword needs a whe...,George R.R. Martin,/author/George-R-R-Martin,"books, mind"


In [12]:
df

,text,author,author_url,tags
0,“The world as we have created it is a process ...,Albert Einstein,/author/Albert-Einstein,"change, deep-thoughts, thinking, world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,/author/J-K-Rowling,"abilities, choices"
2,“There are only two ways to live your life. On...,Albert Einstein,/author/Albert-Einstein,"inspirational, life, live, miracle, miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,/author/Jane-Austen,"aliteracy, books, classic, humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,/author/Marilyn-Monroe,"be-yourself, inspirational"
...,...,...,...,...
95,“You never really understand a person until yo...,Harper Lee,/author/Harper-Lee,better-life-empathy
96,“You have to write the book that wants to be w...,Madeleine L'Engle,/author/Madeleine-LEngle,"books, children, difficult, grown-ups, write, ..."
97,“Never tell the truth to people who are not wo...,Mark Twain,/author/Mark-Twain,truth
98,"“A person's a person, no matter how small.”",Dr. Seuss,/author/Dr-Seuss,inspirational
